<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Isomap: La Hormiga y la Alfombra Enrollada 🐜
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/05%20-%20Reduccion%20de%20Dimensionalidad/Para%20Dummies/02_Isomap_Variedades_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 02 (Isomap y variedades)](../02_Isomap_y_Variedades.ipynb). A veces los datos, aunque tengan muchas columnas, **en realidad viven sobre una superficie delgada** que está doblada o enrollada. *Isomap* intenta **desenrollar esa superficie** midiendo las distancias **como las mediría una hormiga que camina sobre ella**, no como un pájaro que vuela en línea recta.

Al terminar podrás explicar:
1. Qué es una **variedad** («una hoja doblada») y por qué la distancia **por la superficie** puede ser muy distinta de la **recta**.
2. Cómo **Isomap** construye un «mapa de pasos de hormiga» y lo aplana.
3. El peligro del **atajo** (elegir mal el número de vecinos).
4. **Cuándo usar** PCA, Isomap o t-SNE.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Isomap_y_Variedades.ipynb). Antes: [PCA (Dummies)](00_PCA_Dummies.ipynb) y [t-SNE (Dummies)](01_tSNE_Dummies.ipynb).

---
## 1. Una alfombra enrollada 🧶

Imagina una **alfombra** (una superficie plana, de dos dimensiones: largo y ancho) **enrollada** como un rollo de canela. Está dentro de nuestro mundo de **tres dimensiones**, pero **cada punto de la alfombra solo necesita dos números** para ubicarse: «cuánto he avanzado a lo largo» y «cuánto a lo ancho». Los datos «reales» suelen ser así: muchas columnas, pero **pocas causas de fondo**.

Ahora, dos puntos en **capas vecinas** del rollo. Un **pájaro** que vuela en línea recta los ve **muy cerca** (están una capa encima de la otra). Una **hormiga** que solo puede caminar sobre la alfombra tiene que **recorrer todo el rollo** para llegar de uno a otro: para ella están **muy lejos**. La distancia de la hormiga se llama **geodésica**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

X, t = make_swiss_roll(n_samples=1000, noise=0.05, random_state=42)          # la "alfombra enrollada" (rollo suizo)
iso = Isomap(n_neighbors=10).fit(X)
D_hormiga = iso.dist_matrix_                                                 # distancias "caminando por la alfombra"
D_pajaro = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)             # distancias en linea recta

# Un par de puntos que el pajaro ve cerca pero la hormiga ve muy lejos
candidatos = (D_pajaro > 2) & (D_pajaro < 8)
razon = np.where(candidatos, D_hormiga / np.maximum(D_pajaro, 1e-9), 0)
i, j = np.unravel_index(razon.argmax(), razon.shape)
print(f"Pájaro (línea recta): {D_pajaro[i, j]:.1f}   |   Hormiga (por la alfombra): {D_hormiga[i, j]:.1f}   → ¡{D_hormiga[i, j] / D_pajaro[i, j]:.0f} veces más lejos!")

fig = plt.figure(figsize=(14, 5.4))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="Spectral", s=8); ax.view_init(elev=7, azim=-80)
ax.set_title("La alfombra enrollada (3 dimensiones)", fontweight="bold")
ax = fig.add_subplot(1, 2, 2)
ax.scatter(X[:, 0], X[:, 2], c="lightgray", s=8)
ax.plot(*X[[i, j]][:, [0, 2]].T, "o--", color="#2563eb", lw=3, ms=9, label="pájaro (recta)")
ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("z"); ax.legend()
ax.set_title("Vista desde la punta del rollo: dos puntos en capas vecinas", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 2. Isomap: medir con los pasos de la hormiga 🐜

*Isomap* hace tres cosas sencillas:

1. **Une a cada punto con sus vecinos más cercanos** (como cuando cada hormiga solo da pasos cortos hasta la hormiga más próxima). Eso arma una **red de caminitos**.
2. Calcula la distancia entre **cualquier par de puntos** como **la suma de pasos del camino más corto** por esa red (así se «imita» el recorrido de la hormiga por la alfombra).
3. **Dibuja un mapa plano** donde las distancias en línea recta **coincidan lo mejor posible** con esas distancias de hormiga.

Resultado: la alfombra se **desenrolla**. *PCA*, que solo **proyecta con rectas**, en cambio aplasta el rollo y **mezcla las capas**.

In [ ]:
Y_pca = PCA(n_components=2).fit_transform(X)
Y_iso = Isomap(n_neighbors=10, n_components=2).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].scatter(Y_pca[:, 0], Y_pca[:, 1], c=t, cmap="Spectral", s=8); axes[0].set_title("PCA: aplasta el rollo (las capas se mezclan)", fontweight="bold")
axes[1].scatter(Y_iso[:, 0], Y_iso[:, 1], c=t, cmap="Spectral", s=8); axes[1].set_title("Isomap: desenrolla la alfombra (colores ordenados)", fontweight="bold")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Los colores indican la posición a lo largo del rollo: en Isomap van en orden, de un extremo al otro; en PCA se enciman.")

---
## 3. El peligro del atajo ⚡

Todo depende de **a cuántos vecinos** se conecta cada punto (el número *k*):

- **Muy pocos vecinos (k pequeño):** la red queda en **islas separadas**, sin caminos entre ellas. La hormiga no puede cruzar y el mapa se rompe.
- **Muchos vecinos (k grande):** algún «vecino» queda en **otra capa** del rollo, y se crea un **atajo** (un puente que la hormiga jamás podría cruzar en la realidad). Ese solo puente **acorta muchísimos caminos** y el mapa **se dobla sobre sí mismo**.
- **En el punto medio:** el mapa sale bien. Con **ruido** (datos imprecisos) el punto medio se hace **más estrecho**.

Probemos con tres valores.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, k in zip(axes, [5, 10, 30]):
    Y = Isomap(n_neighbors=k, n_components=2).fit_transform(X)
    ax.scatter(Y[:, 0], Y[:, 1], c=t, cmap="Spectral", s=8)
    ax.set_title(f"k = {k} vecinos: " + ("bien (desenrollado)" if k < 20 else "¡atajos! el mapa se dobla"), fontweight="bold"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Con k = 30 aparecen puentes entre capas y los colores ya no van en orden: un número de vecinos demasiado grande rompe el desenrollado.")

---
##### 🛠️ Práctica 1: Cuenta las islas de la red de vecinos

En la sección 3 dijimos que con **muy pocos vecinos** la red de caminitos se parte en **islas** y la hormiga no puede cruzar. Vamos a **contar las islas** de verdad con los puntos `X` de la alfombra enrollada:

1. Para cada número de vecinos `k` en `[1, 3, 10]`, construye la red con `kneighbors_graph(X, n_neighbors=k)`.
2. Cuenta cuántas islas tiene esa red con `connected_components(red, directed=False)` (devuelve el número de islas y a qué isla pertenece cada punto; quédate con el primero).
3. Guarda los resultados en un diccionario `islas_por_k` (por ejemplo `{1: ..., 3: ..., 10: ...}`) e imprímelo.
4. Comprueba que con `k = 1` hay **más de una** isla y que con `k = 10` hay **una sola** (la hormiga llega a todas partes).

In [ ]:
# Escribe tu código aquí

from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import connected_components

islas_por_k = {}
# for k in [1, 3, 10]:
#     red = kneighbors_graph(X, n_neighbors=k)
#     cuantas_islas, _ = connected_components(red, directed=False)
#     islas_por_k[k] = ...

# print(islas_por_k)
# assert islas_por_k[1] > 1
# assert islas_por_k[10] == ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import connected_components

islas_por_k = {}
for k in [1, 3, 10]:
    red = kneighbors_graph(X, n_neighbors=k)                          # la red de caminitos entre vecinos
    cuantas_islas, _ = connected_components(red, directed=False)      # cuantas piezas separadas quedan
    islas_por_k[k] = cuantas_islas

print(islas_por_k)
assert islas_por_k[1] > 1
assert islas_por_k[10] == 1
print("Con 1 vecino la red se rompe en muchas islas; con 10 todo queda conectado y la hormiga puede llegar de cualquier punto a cualquier otro.")
```
</details>

---


---
## 4. ¿Cuál uso? 🧭

| | **PCA** 📷 | **Isomap** 🐜 | **t-SNE** 🎉 |
|---|---|---|---|
| **Analogía** | La mejor **sombra** (foto desde el mejor ángulo) | La **hormiga** que desenrolla una alfombra | El **mapa de amigos** de una fiesta |
| **Qué cuida** | Los datos más dispersos | Las distancias **por la superficie** (panorama global) | **Quién está cerca de quién** |
| **¿Siempre da lo mismo?** | Sí | Sí | **No** (depende de la semilla) |
| **¿Sirve para datos nuevos?** | **Sí** | Sí (de forma aproximada) | **No** |
| **Úsalo cuando...** | Quieres comprimir, acelerar, un punto de partida rápido | Los datos son una **superficie suave y bien cubierta** y te interesa su forma global | Quieres **ver grupos** en datos complicados |
| **Cuidado con...** | Las formas curvas y las escalas distintas | Los **atajos** y las islas (elegir *k*) | Interpretar tamaños y distancias |

---
## Resumen en una frase 🎯

> ***Isomap* desenrolla datos que viven sobre una superficie doblada midiendo las distancias como lo haría una hormiga que camina sobre ella (por una red de vecinos) y dibujando un mapa plano que las respete; funciona muy bien si eliges bien el número de vecinos, y falla si hay islas o atajos.**

### Lo que aprendiste hoy:
- ✅ Muchos datos viven sobre una **superficie doblada** (variedad): pocas causas de fondo, muchas columnas.
- ✅ La distancia **por la superficie** (geodésica) puede ser muy distinta de la **recta**.
- ✅ *Isomap* = **red de vecinos + caminos más cortos + mapa plano**.
- ✅ El número de vecinos *k*: **pocos** → islas; **muchos** → **atajos** que doblan el mapa.
- ✅ **PCA** (sombra), **Isomap** (hormiga) y **t-SNE** (fiesta) sirven para cosas distintas.

> 🎓 **Fin del módulo:** practica todo en el taller del módulo, [`../../homeworks/05_Reduccion_Dimensionalidad_Hands_On.ipynb`](../../homeworks/05_Reduccion_Dimensionalidad_Hands_On.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../02_Isomap_y_Variedades.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué una hormiga y un pájaro pueden medir distancias muy distintas entre los mismos dos puntos de una alfombra enrollada?
2. Aplicas *Isomap* con 40 vecinos a unos datos con mucho ruido y el mapa sale «doblado». ¿Qué crees que pasó y qué cambiarías?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>